In [27]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
from typing import Tuple, List, Union

## 0. Types de Données et Fonctions Utilitaires

### 0.1 Introduction aux Types de Données

On considère un ensemble de candidats/objets évalués selon plusieurs critères. Chaque candidat $c$ a une valuation $v_c(i)$ pour chaque critère $i$.

**Notation:**
- Comparaison $x \succ y$ : l'objet $x$ est préféré à $y$
- Différence: $\text{diff}(x, y)[i] = v_x(i) - v_y(i)$ pour le critère $i$
- **Pros**: critères où $x$ est supérieur à $y$ (différence positive)
- **Cons**: critères où $x$ est inférieur à $y$ (différence négative)

### 0.2 Imports et Configuration

In [28]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import numpy as np
from typing import Tuple, List, Union, Dict

### 0.3 Fonctions Utilitaires

In [97]:
def def_pros_cons_neutrals(differences: Dict[str, float]) -> Tuple[List[str], List[str], List[str]]:
    """
    Identifie les critères positifs (pros), négatifs (cons) et neutres.
    
    Args:
        differences: Dict[str, float] - différences de valuation pour chaque critère
    
    Returns:
        Tuple[List, List, List] - (pros, cons, neutrals)
    """
    pros = []
    cons = []
    neutrals = []
    
    for critere, valeur in differences.items():
        if valeur > 0:
            pros.append(critere)
        elif valeur < 0:
            cons.append(critere)
        else:
            neutrals.append(critere)
    
    return pros, cons, neutrals


def def_trade_offs(differences: Dict[str, float]) -> List[List[str]]:
    """
    Calcule tous les trade-offs possibles (paires pro-con avec somme positive).
    
    Un trade-off (i, j) est valide si: valuation[pro] + valuation[con] > 0
    
    Args:
        differences: Dict[str, float] - différences de valuation
    
    Returns:
        List[List[str, str]] - liste de paires [pro, con]
    """
    pros, cons, _ = def_pros_cons_neutrals(differences)
    trade_offs = []
    
    for pro in pros:
        for con in cons:
            if differences[pro] + differences[con] > 0:
                trade_offs.append([pro, con])
    
    return trade_offs


def compute_difference(candidat_x: Dict[str, float], candidat_y: Dict[str, float]) -> Dict[str, float]:
    """
    Calcule les différences de valuation entre deux candidats.
    
    Args:
        candidat_x: Dict[str, float] - valuations du candidat x
        candidat_y: Dict[str, float] - valuations du candidat y
    
    Returns:
        Dict[str, float] - différences x - y pour chaque critère
    """
    return {k: candidat_x[k] - candidat_y[k] for k in candidat_x.keys()}


print("Fonctions utilitaires chargées ✓")

Fonctions utilitaires chargées ✓


---
# Question 1: trade-offs (1-1) - Une Comparaison x ≻ y

## 1.1 Données d'Entrée

On considère deux candidats x et y évalués selon 7 critères (A, B, C, D, E, F, G).

In [30]:
# Données pour la Question 1
diff_q1 = {
    "A" : 32,
    "B" : 0,
    "C" : -28,
    "D" : 36,
    "E" : 48,
    "F" : -35,
    "G" : -42
}

# Afficher les données
df_q1 = pd.DataFrame(list(diff_q1.items()), columns=['Critère', 'Différence (x - y)'])
print("Question 1 - Données de différence (x - y):")
print(df_q1.to_string(index=False))

# Identifier pros et cons
pros_q1, cons_q1, neutrals_q1 = def_pros_cons_neutrals(diff_q1)
print(f"\nClassification:")
print(f"  Pros (différence positive):  {pros_q1}")
print(f"  Cons (différence négative):  {cons_q1}")
if neutrals_q1:
    print(f"  Neutres:                    {neutrals_q1}")

Question 1 - Données de différence (x - y):
Critère  Différence (x - y)
      A                  32
      B                   0
      C                 -28
      D                  36
      E                  48
      F                 -35
      G                 -42

Classification:
  Pros (différence positive):  ['A', 'D', 'E']
  Cons (différence négative):  ['C', 'F', 'G']
  Neutres:                    ['B']


## 1.2 Calcul des Trade-offs

Un **trade-off** est une paire (i,j) de type (pro, con) où la somme des contributions est positive:
$$\text{trade-off}(i, j) \text{ est valide si } diff[i] + diff[j] > 0$$

Cela signifie que l'avantage du pro compense (au moins partiellement) l'inconvénient du con.

In [31]:
# Calculer les trade-offs valides
trade_offs_q1 = def_trade_offs(diff_q1)

print(f"Trade-offs valides trouvés: {len(trade_offs_q1)}\n")
trade_offs_df_q1 = pd.DataFrame(
    [[pro, con, diff_q1[pro], diff_q1[con], 
      diff_q1[pro] + diff_q1[con]] 
     for pro, con in trade_offs_q1],
    columns=['Pro', 'Con', 'Diff(Pro)', 'Diff(Con)', 'Somme']
)
print(trade_offs_df_q1.to_string(index=False))

Trade-offs valides trouvés: 6

Pro Con  Diff(Pro)  Diff(Con)  Somme
  A   C         32        -28      4
  D   C         36        -28      8
  D   F         36        -35      1
  E   C         48        -28     20
  E   F         48        -35     13
  E   G         48        -42      6


## 1.3 Formulation du Programme Linéaire

### Théorie Mathématique

**Objectif:** Trouver une explication (1-1) de $x \succ y$, c'est-à-dire une bijection qui apparie chaque con de $y$ à un pro de $x$ compensateur.

**Variables de décision:**
$$z_{ij} \in \{0, 1\} \quad \text{pour chaque trade-off valide } (i, j)$$
où $z_{ij} = 1$ si le trade-off $(i, j)$ est sélectionné dans l'explication.

**Fonction objectif:**
$$\text{Maximiser} \sum_{(i,j)} z_{ij}$$

**Contraintes:**
1. **Couverture**: Chaque con doit être couvert exactement une fois
$$\sum_{i \text{ tel que } (i,j) \text{ valide}} z_{ij} = 1 \quad \forall j \in \text{Cons}$$

2. **Unicité**: Chaque pro peut être utilisé au plus une fois
$$\sum_{j \text{ tel que } (i,j) \text{ valide}} z_{ij} \leq 1 \quad \forall i \in \text{Pros}$$

## 1.4 Implémentation avec Gurobi

In [98]:
def exist_explication_1_1_avec_solveur(differences: Dict[str, float]) -> Tuple[bool, Union[List, str]]:
    """
    Formule et résout un programme linéaire pour trouver une explication 1-1.
    
    Args:
        differences: Dict[str, float] - différences de valuation
    
    Returns:
        Tuple[bool, Union[List[List], str]]:
            - (True, explication) si une solution existe
            - (False, certificat) sinon
    """
    pros, cons, _ = def_pros_cons_neutrals(differences)
    trade_offs = def_trade_offs(differences)
    
    # Cas trivial
    if len(cons) == 0:
        return True, []  # Pas de cons à couvrir
    
    if len(trade_offs) == 0:
        certificat = f"Aucun trade-off valide n'existe. Cons à couvrir: {cons}"
        return False, certificat
    
    try:
        # === CRÉER LE MODÈLE GUROBI ===
        model = gp.Model("Explication_1_1")
        model.setParam('OutputFlag', 0)  # Désactiver l'affichage détaillé
        
        # === VARIABLES DE DÉCISION ===
        z = {}
        for i, (pro, con) in enumerate(trade_offs):
            z[i] = model.addVar(vtype=GRB.BINARY, name=f"z_{pro}_{con}")
        
        # === FONCTION OBJECTIF ===
        model.setObjective(gp.quicksum(z[i] for i in range(len(trade_offs))), GRB.MAXIMIZE)
        
        # === CONTRAINTE 1: Couverture des cons ===
        for con in cons:
            indices_con = [i for i, (pro, c) in enumerate(trade_offs) if c == con]
            if indices_con:
                model.addConstr(
                    gp.quicksum(z[i] for i in indices_con) == 1,
                    name=f"Couverture_{con}"
                )
            else:
                certificat = f"Le critère négatif '{con}' ne peut être couvert par aucun trade-off valide."
                return False, certificat
        
        # === CONTRAINTE 2: Unicité des pros ===
        for pro in pros:
            indices_pro = [i for i, (p, con) in enumerate(trade_offs) if p == pro]
            if indices_pro:
                model.addConstr(
                    gp.quicksum(z[i] for i in indices_pro) <= 1,
                    name=f"Unicite_{pro}"
                )
        
        # === RÉSOUDRE LE PROBLÈME ===
        model.optimize()
        
        # === ANALYSER LES RÉSULTATS ===
        if model.status == GRB.OPTIMAL:
            explication = []
            for i, (pro, con) in enumerate(trade_offs):
                if z[i].X > 0.5:
                    explication.append([pro, con])
            
            cons_couverts = set(con for pro, con in explication)
            if len(cons_couverts) == len(cons):
                return True, explication
            else:
                cons_non_couverts = set(cons) - cons_couverts
                certificat = f"Impossible de couvrir tous les cons. Non couverts: {list(cons_non_couverts)}"
                return False, certificat
        
        elif model.status == GRB.INFEASIBLE:
            certificat = "\n🔴 CERTIFICAT DE NON-EXISTENCE\n"
            certificat += "="*50 + "\n"
            certificat += f"Le modèle est infaisable.\n"
            certificat += f"Il n'existe PAS d'explication 1-1 qui couvre tous les {len(cons)} cons.\n\n"
            certificat += f"Analyse:\n"
            certificat += f"  • Pros disponibles: {len(pros)} ({pros})\n"
            certificat += f"  • Cons à couvrir:  {len(cons)} ({cons})\n"
            certificat += f"  • Trade-offs:      {len(trade_offs)}\n\n"
            certificat += f"Raisons possibles:\n"
            certificat += f"  1. Pas assez de pros pour couvrir tous les cons\n"
            certificat += f"  2. Certains cons ne peuvent être compensés\n"
            
            try:
                model.computeIIS()
                iis_constrs = [c.ConstrName for c in model.getConstrs() if c.IISConstr]
                if iis_constrs:
                    certificat += f"  3. Conflits identifiés par Gurobi:\n"
                    for constr_name in iis_constrs:
                        certificat += f"     - {constr_name}\n"
            except:
                pass
            
            certificat += "\n" + "="*50
            return False, certificat
        
        else:
            certificat = f"Le solveur n'a pas convergé. Statut: {model.status}"
            return False, certificat
    
    except gp.GurobiError as e:
        return False, f"Erreur Gurobi: {str(e)}"

print("Fonction 'exist_explication_1_1_avec_solveur' définie ✓")

Fonction 'exist_explication_1_1_avec_solveur' définie ✓


## 1.5 Résolution - Question 1

In [33]:
# Résoudre pour la Question 1
existe_q1, resultat_q1 = exist_explication_1_1_avec_solveur(diff_q1)

print("\n" + "="*70)
print("QUESTION 1 - RÉSULTATS DE L'OPTIMISATION")
print("="*70)

if existe_q1:
    print("\n✅ EXPLICATION 1-1 TROUVÉE\n")
    print(f"Nombre de trade-offs sélectionnés: {len(resultat_q1)}\n")
    
    resultats_df_q1 = pd.DataFrame(
        [[pro, con, diff_q1[pro], diff_q1[con], 
          diff_q1[pro] + diff_q1[con]] 
         for pro, con in resultat_q1],
        columns=['Pro', 'Con', 'Diff(Pro)', 'Diff(Con)', 'Somme']
    )
    print(resultats_df_q1.to_string(index=False))
    
    # Vérification
    cons_couverts_q1 = sorted(set(con for pro, con in resultat_q1))
    print(f"\nCons couverts: {cons_couverts_q1}")
    print(f"Tous les cons couverts: {'Oui ✓' if len(cons_couverts_q1) == len(cons_q1) else 'Non ✗'}")
else:
    print(resultat_q1)


QUESTION 1 - RÉSULTATS DE L'OPTIMISATION

✅ EXPLICATION 1-1 TROUVÉE

Nombre de trade-offs sélectionnés: 3

Pro Con  Diff(Pro)  Diff(Con)  Somme
  A   C         32        -28      4
  D   F         36        -35      1
  E   G         48        -42      6

Cons couverts: ['C', 'F', 'G']
Tous les cons couverts: Oui ✓


## 1.6 Preuve Mathématique - Question 1

### Propriétés de l'Explication (1-1)

**Définition Formelle:**
Une explication (1-1) de $x \succ y$ est une bijection $f: \text{Cons}(y) \to P \subseteq \text{Pros}(x)$ telle que:
$$\forall j \in \text{Cons}(y): \text{diff}(f(j)) + \text{diff}(j) > 0$$

**Caractérisation du Problème:**

1. **Ensemble Pros**: Les critères où $x$ surpasse $y$ (différence positive)
2. **Ensemble Cons**: Les critères où $y$ surpasse $x$ (différence négative)
3. **Trade-offs valides**: Paires $(i, j)$ telles que $\text{diff}[i] + \text{diff}[j] > 0$

**Formulation LP (MILP):**
$$\begin{align}
\text{Maximiser} \quad & \sum_{(i,j)} z_{ij} \\
\text{t.q.} \quad & \sum_{i: (i,j) \text{ valide}} z_{ij} = 1 \quad \forall j \in \text{Cons} \\
& \sum_{j: (i,j) \text{ valide}} z_{ij} \leq 1 \quad \forall i \in \text{Pros} \\
& z_{ij} \in \{0, 1\}
\end{align}$$

**Optimalité:**
- Si la solution est **réalisable**, elle maximise le nombre de trade-offs couverts
- Si la solution est **infaisable**, Gurobi le certifie et fournit un IIS (Irreducible Inconsistent Subsystem)

---
# Question 2: Analyse Comparative - Classement de 8 Candidats

## 2.1 Données d'Entrée

On considère 8 candidats (x, y, z, t, u, v, w, w′) évalués selon 7 critères avec des poids respectifs.

In [34]:
# Données pour la Question 2
candidats_q2 = {
    'x': {'A': 85, 'B': 81, 'C': 71, 'D': 69, 'E': 75, 'F': 81, 'G': 88},
    'y': {'A': 81, 'B': 81, 'C': 75, 'D': 63, 'E': 67, 'F': 88, 'G': 95},
    'z': {'A': 74, 'B': 89, 'C': 74, 'D': 81, 'E': 68, 'F': 84, 'G': 79},
    't': {'A': 74, 'B': 71, 'C': 84, 'D': 91, 'E': 77, 'F': 76, 'G': 73},
    'u': {'A': 72, 'B': 75, 'C': 66, 'D': 85, 'E': 88, 'F': 66, 'G': 93},
    'v': {'A': 71, 'B': 73, 'C': 63, 'D': 92, 'E': 76, 'F': 79, 'G': 93},
    'w': {'A': 79, 'B': 69, 'C': 78, 'D': 76, 'E': 67, 'F': 84, 'G': 79},
    "w'": {'A': 57, 'B': 76, 'C': 81, 'D': 76, 'E': 82, 'F': 86, 'G': 77}
}

# Poids des critères
weights = {'A': 8, 'B': 7, 'C': 7, 'D': 6, 'E': 6, 'F': 5, 'G': 6}

# Afficher les données
df_q2_candidats = pd.DataFrame(candidats_q2).T
print("Question 2 - Valuations des 8 candidats:")
print(df_q2_candidats)
print(f"\nPoids des critères: {weights}")

Question 2 - Valuations des 8 candidats:
     A   B   C   D   E   F   G
x   85  81  71  69  75  81  88
y   81  81  75  63  67  88  95
z   74  89  74  81  68  84  79
t   74  71  84  91  77  76  73
u   72  75  66  85  88  66  93
v   71  73  63  92  76  79  93
w   79  69  78  76  67  84  79
w'  57  76  81  76  82  86  77

Poids des critères: {'A': 8, 'B': 7, 'C': 7, 'D': 6, 'E': 6, 'F': 5, 'G': 6}


### Classement Donné
$$x \succ y \succ z \succ t \succ u \succ v \succ w \succ w'$$

### Montrer que w ≻ w' n'est pas justifiable par une explication (1-1)

Pour `w` contre `w′`, la différence `w - w′` donne :
- Pros (différence positive) : A, G  
- Cons (différence négative) : B, C, E, F  
- Neutres : D

L’explication (1-1) exige **au moins autant de pros que de cons** pour apparier chaque con à un pro distinct. Ici, on a seulement 2 pros pour 4 cons (**len(pros)=2 < len(cons)=4**). Il est donc impossible de construire une correspondance 1-1 couvrant tous les cons : le modèle Gurobi marque la comparaison `w ≻ w′` comme infaisable pour cette raison structurelle (manque de pros).

## 2.2 Formulation du programme linéaire ($1-m$)

**Variables de décision (binaires)**  
- $y_p = 1$ si le pro $p$ est utilisé. 
- $x_{p,c} = 1$ si le con $c$ est affecté au pro $p$.

**Données**  
- $d_i$ : différence (éventuellement pondérée) sur le critère $i$ entre $x$ et $y$.  
  Pros $\Rightarrow d_i > 0$, Cons $\Rightarrow d_i < 0$.

**Objectif (exemple)**  
Maximiser la marge totale :
$$\max \sum_{p\in pros} d_p\,y_p + \sum_{p\in pros}\sum_{c\in cons} d_c\, x_{p,c}$$

**Contraintes**
1. *Couverture des cons* (chacun exactement une fois) :
$$\sum_{p\in pros} x_{p,c} = 1 \quad \forall c \in cons$$
2. *Activation cohérente* (on n’assigne pas si le pro n’est pas choisi) :
$$x_{p,c} \le y_p \quad \forall p\in pros,\; \forall c\in cons$$
3. *Trade-off positif pour chaque pro actif* :
$$d_p\,y_p + \sum_{c\in cons} d_c\, x_{p,c} \ge \varepsilon\, y_p \quad (\varepsilon > 0 \text{ petit})$$
   Un pro ne peut être actif ($y_p=1$) que si la somme de ses cons affectés est plus que compensée.

**Certificat de non-existence**  
Si le modèle est infaisable, le solveur retourne un IIS (Irreducible Inconsistent Subsystem) qui identifie les contraintes en conflit (par ex. trop peu de pros ou cons trop pénalisants).

In [43]:
def exist_explication_1_m_avec_solveur(differences: Dict[str, float], epsilon: float = 1e-3) -> Tuple[bool, Union[List[Dict[str, List[str]]], str]]:
    """
    Programme linéaire (1-m) : un pro peut compenser plusieurs cons.

    Args:
        differences: Dict[str, float] - différences (déjà pondérées si besoin) entre x et y.
        epsilon: float - marge minimale pour qu'un pro soit considéré compensateur.

    Returns:
        (True, explication) si une solution existe, où explication est une liste de dicts
        [{'pro': p, 'cons': [c1, c2, ...]}, ...]
        (False, certificat) sinon.
    """
    pros, cons, _ = def_pros_cons_neutrals(differences)

    # Cas trivial
    if len(cons) == 0:
        return True, []
    if len(pros) == 0:
        return False, "Aucun pro disponible pour compenser les cons."

    try:
        model = gp.Model("Explication_1_m")
        model.setParam('OutputFlag', 0)

        # Variables
        y = {p: model.addVar(vtype=GRB.BINARY, name=f"y_{p}") for p in pros}
        x = {(p, c): model.addVar(vtype=GRB.BINARY, name=f"x_{p}_{c}") for p in pros for c in cons}

        # Objectif : maximiser la marge totale
        model.setObjective(
            gp.quicksum(differences[p] * y[p] for p in pros)
            + gp.quicksum(differences[c] * x[(p, c)] for p in pros for c in cons),
            GRB.MAXIMIZE,
        )

        # Couverture des cons
        for c in cons:
            model.addConstr(gp.quicksum(x[(p, c)] for p in pros) == 1, name=f"couverture_{c}")

        # Lien activation
        for p in pros:
            for c in cons:
                model.addConstr(x[(p, c)] <= y[p], name=f"activation_{p}_{c}")

        # Trade-off positif pour chaque pro actif
        for p in pros:
            model.addConstr(
                differences[p] * y[p] + gp.quicksum(differences[c] * x[(p, c)] for c in cons) >= epsilon * y[p],
                name=f"positivite_{p}",
            )

        model.optimize()

        if model.status == GRB.OPTIMAL:
            explication = []
            for p in pros:
                if y[p].X > 0.5:
                    cons_affectes = [c for c in cons if x[(p, c)].X > 0.5]
                    explication.append({"pro": p, "cons": cons_affectes})
            return True, explication

        elif model.status == GRB.INFEASIBLE:
            certificat = "\nCERTIFICAT DE NON-EXISTENCE (1-m)\n" + "=" * 50 + "\n"
            certificat += "Le modèle est infaisable : impossible de couvrir tous les cons avec des paquets (1-m).\n"
            try:
                model.computeIIS()
                iis_constrs = [c.ConstrName for c in model.getConstrs() if c.IISConstr]
                if iis_constrs:
                    certificat += "Contraintes en conflit (IIS) :\n"
                    for cname in iis_constrs:
                        certificat += f"  - {cname}\n"
            except:
                pass
            return False, certificat

        else:
            return False, f"Le solveur n'a pas convergé. Statut: {model.status}"

    except gp.GurobiError as e:
        return False, f"Erreur Gurobi: {str(e)}"

## 2.3 Vérification ($1-m$) : u ≻ v et w ≻ w′
Nous testons automatiquement le modèle ($1-m$) :
- montrer qu'aucune explication ($1-m$) n'existe pour $u \succ v$,
- vérifier qu'une explication ($1-m$) existe pour $w \succ w'$.
Les différences sont pondérées par les poids des critères avant d'être passées au solveur.

In [44]:
def comparer_1_m(candidat_x: str, candidat_y: str):
    """Affiche la comparaison (1-m) pondérée entre deux candidats."""
    diff = {k: weights[k] * (candidats_q2[candidat_x][k] - candidats_q2[candidat_y][k]) for k in weights}
    df = pd.DataFrame(
        [(crit, candidats_q2[candidat_x][crit], candidats_q2[candidat_y][crit], diff[crit]) for crit in weights],
        columns=["Critère", candidat_x, candidat_y, "Différence pondérée"],
    )
    print(f"\n=== {candidat_x} ≻ {candidat_y} (1-m) ===")
    print(df.to_string(index=False))
    pros, cons, neutrals = def_pros_cons_neutrals(diff)
    print(f"Pros: {pros} | Cons: {cons} | Neutres: {neutrals}")
    existe, explication = exist_explication_1_m_avec_solveur(diff)
    if existe:
        print("\n✅ Explication (1-m) trouvée")
        for bloc in explication:
            print(f"  - Pro {bloc['pro']} couvre: {bloc['cons']}")
    else:
        print("\n❌ Aucune explication (1-m)")
        print(explication)
    return existe, explication

comparaisons = [("u", "v"), ("w", "w'")]
for a, b in comparaisons:
    comparer_1_m(a, b)


=== u ≻ v (1-m) ===
Critère  u  v  Différence pondérée
      A 72 71                    8
      B 75 73                   14
      C 66 63                   21
      D 85 92                  -42
      E 88 76                   72
      F 66 79                  -65
      G 93 93                    0
Pros: ['A', 'B', 'C', 'E'] | Cons: ['D', 'F'] | Neutres: ['G']

❌ Aucune explication (1-m)

CERTIFICAT DE NON-EXISTENCE (1-m)
Le modèle est infaisable : impossible de couvrir tous les cons avec des paquets (1-m).
Contraintes en conflit (IIS) :
  - couverture_D
  - couverture_F
  - positivite_A
  - positivite_B
  - positivite_C
  - positivite_E


=== w ≻ w' (1-m) ===
Critère  w  w'  Différence pondérée
      A 79  57                  176
      B 69  76                  -49
      C 78  81                  -21
      D 76  76                    0
      E 67  82                  -90
      F 84  86                  -10
      G 79  77                   12
Pros: ['A', 'G'] | Cons: ['B', 'C', 'E', '

## 2.4 Interprétation qualitative (1-m) — u ≻ v et w ≻ w′

- u ≻ v — impossibilité (1-m) :
  - Pros pondérés: A=+8, B=+14, C=+21, E=+72; Cons: D=−42, F=−65; G=0.
  - Positivité locale par pro: tout pro actif doit vérifier d_p + \(\sum d_c\) (cons qui lui sont affectés) ≥ ε.
  - Seul E peut compenser F ou D isolément, mais pas les deux: 72−65−42 = −35 < ε.
  - A/B/C sont trop faibles pour absorber D (−42) ou F (−65).
  - IIS (Gurobi): `couverture_D`, `couverture_F`, `positivite_A`, `positivite_B`, `positivite_C`, `positivite_E` → obligation de couvrir D et F + positivité par pro incompatible ⇒ modèle infaisable.

- w ≻ w′ — existence (1-m) :
  - Pros pondérés: A=+176, G=+12; Cons: B=−49, C=−21, E=−90, F=−10.
  - Le pro A est suffisant pour absorber tous les cons: 176−49−21−90−10 = +6 ≥ ε.
  - Une explication valide est: A → {B, C, E, F}, G → {}.

Conclusion: la non‑existence pour u ≻ v vient d’une capacité par‑pro insuffisante (E ne peut absorber D et F ensemble, les autres pros sont trop faibles), tandis que w ≻ w′ admet une explication (1‑m) grâce à un pro A suffisamment dominant.

---
# Question 3: Trade-offs (m-1)

.
## 3.1 "À la main" — y ≻ z (pondéré)

- Différences pondérées d = w ⊙ (y − z):
  - Pros: A=+56, C=+7, F=+20, G=+96
  - Cons: B=−56, D=−108, E=−6
.
- Principe (m-1): un con peut recevoir plusieurs pros, mais chaque pro ne peut contribuer qu’à un seul con (unicité côté pros). Chaque con doit être rendu positif:
  $$d_c + \sum_{p \in \text{pros affectés à } c} d_p \ge \varepsilon$$
.
- Essais de bundles:
  - Pour D (−108), les combinaisons sans A+G sont insuffisantes, donc on doit utiliser G avec (A ou F):
    - G+F = 96+20 = 116 (OK), ou G+A = 96+56 = 152 (OK).
  - Si on choisit D ← G+F, il reste A (+56) et C (+7) pour B (−56) et E (−6):
    - B nécessite ≥ 56+ε → A seul (=56) n’atteint pas ε; A+C (=63) couvre B, mais il ne reste alors plus de pro pour E.
  - Si on choisit D ← G+A, il reste F (+20) et C (+7): insuffisant pour B (−56) et E (−6).
.
⇒ Aucun partitionnement des pros {A,C,F,G} ne permet de rendre simultanément B, D et E positifs sous la contrainte d’unicité côté pros. Donc, il n’existe pas d’explication (m-1) pour y ≻ z.

## 3.2 Formulation du programme linéaire (m-1)

Objectif: autoriser plusieurs pros pour un même con (m-1), tout en imposant qu’un pro n’aide au plus qu’un seul con.


Variables (binaires):
- $x_{p,c} = 1$ si le pro $p$ est affecté au con $c$; 0 sinon.
- $z_p = 1$ si le pro $p$ est utilisé (au moins pour un con); 0 sinon.


Données:
- $d_i$ = différence pondérée sur le critère $i$ (pros: $d_i>0$, cons: $d_i<0$).
- $\varepsilon>0$ petite marge.


Contraintes:
1. Unicité côté pros (un pro au plus pour un seul con):
   $$\sum_{c\in cons} x_{p,c} \le z_p \quad \text{et} \quad \sum_{c\in cons} x_{p,c} \le 1 \quad \forall p\in pros$$
2. Activation:
   $$x_{p,c} \le z_p \quad \forall p,c$$
3. Positivité par con (chaque con doit être compensé):
   $$d_c + \sum_{p\in pros} d_p\, x_{p,c} \ge \varepsilon \quad \forall c\in cons$$

Fonction objectif (exemples):
- Minimiser le nombre de pros utilisés: $\min \sum_{p} z_p$
- Ou maximiser la marge totale: $\max \sum_{c} (d_c + \sum_p d_p x_{p,c})$.

Certificat de non-existence:
- Si infaisable, on extrait un IIS pour lister les contraintes en conflit (p.ex. certaines positivités de cons + unicité des pros).

In [45]:
def exist_explication_m_1_avec_solveur(differences: Dict[str, float], epsilon: float = 1e-3, objective: str = "min_pros") -> Tuple[bool, Union[List[Dict[str, List[str]]], str]]:
    """
    Programme linéaire (m-1): plusieurs pros peuvent compenser un con;
    chaque pro ne peut être utilisé que pour un seul con.

    Args:
        differences: Dict[str, float] - différences pondérées entre x et y.
        epsilon: float - marge minimale de compensation par con.
        objective: str - 'min_pros' (par défaut) ou 'max_margin'.

    Returns:
        - (True, explication): liste de bundles par con: [{'con': c, 'pros': [p1,p2,...]}, ...]
        - (False, certificat): texte expliquant l'infaisabilité avec IIS si disponible.
    """
    pros, cons, _ = def_pros_cons_neutrals(differences)
    if len(cons) == 0:
        return True, []  # Rien à compenser
    if len(pros) == 0:
        return False, "Aucun pro disponible pour compenser les cons (m-1)."
    try:
        model = gp.Model("Explication_m_1")
        model.setParam('OutputFlag', 0)
        # Variables
        x = {(p, c): model.addVar(vtype=GRB.BINARY, name=f"x_{p}_{c}") for p in pros for c in cons}
        z = {p: model.addVar(vtype=GRB.BINARY, name=f"z_{p}") for p in pros}
        # Unicité côté pros + activation
        for p in pros:
            model.addConstr(gp.quicksum(x[(p, c)] for c in cons) <= 1, name=f"unicite_{p}")
            for c in cons:
                model.addConstr(x[(p, c)] <= z[p], name=f"activation_{p}_{c}")
        # Positivité par con
        for c in cons:
            model.addConstr(differences[c] + gp.quicksum(differences[p] * x[(p, c)] for p in pros) >= epsilon, name=f"positivite_{c}")
        # Objectif
        if objective == "min_pros":
            model.setObjective(gp.quicksum(z[p] for p in pros), GRB.MINIMIZE)
        else:  # max_margin
            model.setObjective(gp.quicksum(differences[c] + gp.quicksum(differences[p] * x[(p, c)] for p in pros) for c in cons), GRB.MAXIMIZE)
        model.optimize()
        if model.status == GRB.OPTIMAL:
            explication = []
            for c in cons:
                pros_affectes = [p for p in pros if x[(p, c)].X > 0.5]
                explication.append({"con": c, "pros": pros_affectes})
            return True, explication
        elif model.status == GRB.INFEASIBLE:
            certificat = "\nCERTIFICAT DE NON-EXISTENCE (m-1)\n" + "=" * 50 + "\n"
            certificat += "Le modèle (m-1) est infaisable: impossibilité de rendre chaque con positif avec des pros uniques.\n"
            try:
                model.computeIIS()
                iis_constrs = [c.ConstrName for c in model.getConstrs() if c.IISConstr]
                if iis_constrs:
                    certificat += "Contraintes en conflit (IIS) :\n"
                    for cname in iis_constrs:
                        certificat += f"  - {cname}\n"
            except:
                pass
            return False, certificat
        else:
            return False, f"Le solveur n'a pas convergé. Statut: {model.status}"
    except gp.GurobiError as e:
        return False, f"Erreur Gurobi: {str(e)}"

In [47]:
# Résolution (m-1) pour y ≻ z
diff_xy = {k: weights[k] * (candidats_q2['y'][k] - candidats_q2['z'][k]) for k in weights}
df = pd.DataFrame([(k, diff_xy[k]) for k in diff_xy], columns=["Critère", "Différence pondérée (y - z)"])
print("\nQuestion 3 — y ≻ z (m-1)")
print(df.to_string(index=False))
pros, cons, _ = def_pros_cons_neutrals(diff_xy)
print(f"Pros: {pros} | Cons: {cons}")
existe_m1_xy, explication_m1_xy = exist_explication_m_1_avec_solveur(diff_xy, epsilon=1e-3, objective="min_pros")
if existe_m1_xy:
    print("\n✅ Explication (m-1) trouvée pour y ≻ z")
    for bloc in explication_m1_xy:
        print(f"  - Con {bloc['con']} couvert par pros: {bloc['pros']}")
else:
    print("\n❌ Aucune explication (m-1) pour y ≻ z")
    print(explication_m1_xy)


Question 3 — y ≻ z (m-1)
Critère  Différence pondérée (y - z)
      A                           56
      B                          -56
      C                            7
      D                         -108
      E                           -6
      F                           20
      G                           96
Pros: ['A', 'C', 'F', 'G'] | Cons: ['B', 'D', 'E']

❌ Aucune explication (m-1) pour y ≻ z

CERTIFICAT DE NON-EXISTENCE (m-1)
Le modèle (m-1) est infaisable: impossibilité de rendre chaque con positif avec des pros uniques.
Contraintes en conflit (IIS) :
  - unicite_A
  - unicite_C
  - unicite_F
  - unicite_G
  - positivite_B
  - positivite_D
  - positivite_E



## 3.3 Variante relâchée (m-1) — Explicabilité améliorée

Constat: la contrainte stricte d_c + ∑ d_p ≥ ε pour **tous** les cons peut être trop restrictive. En réalité, pour expliquer x ≻ y, il suffit qu’au moins un con soit strictement compensé, les autres pouvant être **neutres** (marge = 0).

**Principe relâché:**
- Chaque con doit être non-négatif : d_c + ∑_{p affectés} d_p ≥ 0.
- Au moins un con doit être **strictement positif** : d_c + ∑ d_p ≥ ε (pour au moins un c).

Cela capture l’idée qu’une préférence x ≻ y reste valide même si certains trade-offs sont équilibrés (A+B=0), tant qu’il existe au moins une inégalité stricte en faveur de x.

**Implémentation (via MILP):**
- Variables supplémentaires: s_c ∈ {0,1} indique si le con c a une marge stricte.
- Contraintes:
  - d_c + ∑ d_p ≥ 0 pour tout c (non-négativité)
  - d_c + ∑ d_p ≥ ε s_c pour tout c (si s_c=1, marge stricte)
  - ∑ s_c ≥ 1 (au moins un con strict)
- Objectif: minimiser ∑ z_p (nombre de pros) ou maximiser marge totale.

In [99]:
def exist_explication_m_1_relaxed_avec_solveur(differences: Dict[str, float], epsilon: float = 1e-3, objective: str = "min_pros") -> Tuple[bool, Union[List[Dict[str, List[str]]], str]]:
    """
    Variante relâchée (m-1): chaque con doit être ≥ 0, avec au moins un con strictement > ε.

    Args:
        differences: Dict[str, float] - différences pondérées.
        epsilon: float - marge minimale pour au moins un con.
        objective: str - 'min_pros' ou 'max_margin'.

    Returns:
        - (True, explication): [{'con': c, 'pros': [p1,...], 'margin': float}, ...]
        - (False, certificat): texte avec IIS.
    """
    pros, cons, _ = def_pros_cons_neutrals(differences)
    if len(cons) == 0:
        return True, []
    if len(pros) == 0:
        return False, "Aucun pro disponible (m-1 relâché)."
    try:
        model = gp.Model("Explication_m_1_relaxed")
        model.setParam('OutputFlag', 0)
        # Variables
        x = {(p, c): model.addVar(vtype=GRB.BINARY, name=f"x_{p}_{c}") for p in pros for c in cons}
        z = {p: model.addVar(vtype=GRB.BINARY, name=f"z_{p}") for p in pros}
        s = {c: model.addVar(vtype=GRB.BINARY, name=f"s_{c}") for c in cons}  # strict indicator
        # Unicité + activation
        for p in pros:
            model.addConstr(gp.quicksum(x[(p, c)] for c in cons) <= 1, name=f"unicite_{p}")
            for c in cons:
                model.addConstr(x[(p, c)] <= z[p], name=f"activation_{p}_{c}")
        # Non-négativité pour tous les cons
        for c in cons:
            model.addConstr(differences[c] + gp.quicksum(differences[p] * x[(p, c)] for p in pros) >= 0, name=f"nonneg_{c}")
        # Stricte positivité si s_c=1
        M = sum(abs(differences[k]) for k in differences) + 100  # big-M
        for c in cons:
            model.addConstr(differences[c] + gp.quicksum(differences[p] * x[(p, c)] for p in pros) >= epsilon * s[c], name=f"strict_{c}")
        # Au moins un con strict
        model.addConstr(gp.quicksum(s[c] for c in cons) >= 1, name="at_least_one_strict")
        # Objectif
        if objective == "min_pros":
            model.setObjective(gp.quicksum(z[p] for p in pros), GRB.MINIMIZE)
        else:
            model.setObjective(gp.quicksum(differences[c] + gp.quicksum(differences[p] * x[(p, c)] for p in pros) for c in cons), GRB.MAXIMIZE)
        model.optimize()
        if model.status == GRB.OPTIMAL:
            explication = []
            for c in cons:
                pros_affectes = [p for p in pros if x[(p, c)].X > 0.5]
                margin = differences[c] + sum(differences[p] for p in pros_affectes)
                explication.append({"con": c, "pros": pros_affectes, "margin": round(margin, 3)})
            return True, explication
        elif model.status == GRB.INFEASIBLE:
            certificat = "\nCERTIFICAT DE NON-EXISTENCE (m-1 relâché)\n" + "=" * 50 + "\n"
            certificat += "Même en autorisant des marges nulles, aucun assignement valide.\n"
            try:
                model.computeIIS()
                iis_constrs = [c.ConstrName for c in model.getConstrs() if c.IISConstr]
                if iis_constrs:
                    certificat += "Contraintes en conflit (IIS) :\n"
                    for cname in iis_constrs:
                        certificat += f"  - {cname}\n"
            except:
                pass
            return False, certificat
        else:
            return False, f"Statut: {model.status}"
    except gp.GurobiError as e:
        return False, f"Erreur Gurobi: {str(e)}"

In [95]:
# Test variante relâchée (m-1) pour y ≻ z
diff_yz = {k: weights[k] * (candidats_q2['y'][k] - candidats_q2['z'][k]) for k in weights}
df_yz = pd.DataFrame([(k, diff_yz[k]) for k in diff_yz], columns=["Critère", "Différence pondérée (y - z)"])
print("\nQuestion 3 (relâché) — y ≻ z (m-1)")
print(df_yz.to_string(index=False))
pros_yz, cons_yz, _ = def_pros_cons_neutrals(diff_yz)
print(f"Pros: {pros_yz} | Cons: {cons_yz}")

existe_m1_yz_rel, explication_m1_yz_rel = exist_explication_m_1_relaxed_avec_solveur(diff_yz, epsilon=1e-3, objective="min_pros")
if existe_m1_yz_rel:
    print("\n✅ Explication (m-1 relâchée) trouvée pour y ≻ z")
    for bloc in explication_m1_yz_rel:
        print(f"  - Con {bloc['con']} couvert par {bloc['pros']} → marge = {bloc['margin']}")
else:
    print("\n❌ Aucune explication (m-1 relâchée) pour y ≻ z")
    print(explication_m1_yz_rel)


Question 3 (relâché) — y ≻ z (m-1)
Critère  Différence pondérée (y - z)
      A                           56
      B                          -56
      C                            7
      D                         -108
      E                           -6
      F                           20
      G                           96
Pros: ['A', 'C', 'F', 'G'] | Cons: ['B', 'D', 'E']

✅ Explication (m-1 relâchée) trouvée pour y ≻ z
  - Con B couvert par ['A'] → marge = 0
  - Con D couvert par ['F', 'G'] → marge = 8
  - Con E couvert par ['C'] → marge = 1


---
# Question 4: Combinaison Trade-offs (m-1) & (1-m)

.
## 4.0 Etude de la comparaison z ≻ t avec les méthodes précédentes

In [51]:
# Question 4 — Tests d’inexistence pour z ≻ t (1-m et m-1)
diff_zt = {k: weights[k] * (candidats_q2['z'][k] - candidats_q2['t'][k]) for k in weights}
df_zt = pd.DataFrame([(k, diff_zt[k]) for k in diff_zt], columns=["Critère", "Différence pondérée (z - t)"])
print("\nQuestion 4 — z ≻ t")
print(df_zt.to_string(index=False))
pros_zt, cons_zt, _ = def_pros_cons_neutrals(diff_zt)
print(f"Pros: {pros_zt} | Cons: {cons_zt}")

# Test 1-m
existe_1m_zt, exp_1m_zt = exist_explication_1_m_avec_solveur(diff_zt, epsilon=1e-3)
if existe_1m_zt:
    print("\n⚠️ Explication (1-m) trouvée — ce serait contradictoire avec l’énoncé")
    print(exp_1m_zt)
else:
    print("\n❌ Aucune explication (1-m) pour z ≻ t")
    print(exp_1m_zt)

# Test m-1 (strict)
existe_m1_zt, exp_m1_zt = exist_explication_m_1_avec_solveur(diff_zt, epsilon=1e-3, objective="min_pros")
if existe_m1_zt:
    print("\n⚠️ Explication (m-1) trouvée — ce serait contradictoire avec l’énoncé")
    print(exp_m1_zt)
else:
    print("\n❌ Aucune explication (m-1) pour z ≻ t")
    print(exp_m1_zt)


Question 4 — z ≻ t
Critère  Différence pondérée (z - t)
      A                            0
      B                          126
      C                          -70
      D                          -60
      E                          -54
      F                           40
      G                           36
Pros: ['B', 'F', 'G'] | Cons: ['C', 'D', 'E']

❌ Aucune explication (1-m) pour z ≻ t

CERTIFICAT DE NON-EXISTENCE (1-m)
Le modèle est infaisable : impossible de couvrir tous les cons avec des paquets (1-m).
Contraintes en conflit (IIS) :
  - couverture_C
  - couverture_D
  - positivite_B
  - positivite_F
  - positivite_G


❌ Aucune explication (m-1) pour z ≻ t

CERTIFICAT DE NON-EXISTENCE (m-1)
Le modèle (m-1) est infaisable: impossibilité de rendre chaque con positif avec des pros uniques.
Contraintes en conflit (IIS) :
  - unicite_B
  - unicite_F
  - positivite_C
  - positivite_D
  - positivite_E



## 4.1 Interprétation qualitative — Combinaison $(1\text{-}m)$ + $(m\text{-}1)$ pour $z \succ t$

Différences pondérées $(z - t)$: $B=+126$, $F=+40$, $G=+36$; $C=-70$, $D=-60$, $E=-54$; $A=0$.

- Blocage séparé:
  - En $(1\text{-}m)$, un pro $p$ doit compenser seul ses cons affectés: par exemple $B$ peut absorber $C$ car $126 + (-70) = +56 > 0$, mais $F$ ou $G$ ne peuvent absorber $D$ ou $E$ chacun: $40 + (-60) < 0$, $36 + (-54) < 0$.
  - En $(m\text{-}1)$, chaque con $c$ doit vérifier $d_c + \sum_{p} d_p x_{p,c} \ge \varepsilon$ avec l’unicité côté pros $\sum_{c} x_{p,c} \le 1$. L’unicité empêche de réutiliser $B$ sur plusieurs cons tout en couvrant simultanément $D$ et $E$.

- Idée combinée:
  - Utiliser un pro “fort” ($B$) en mode $(1\text{-}m)$ pour couvrir un sous-ensemble exigeant (par ex. $C$), i.e. $a_{B,C}=1$ et $d_B + d_C \ge \varepsilon$.
  - Utiliser $(m\text{-}1)$ pour d’autres cons avec plusieurs pros plus faibles mais dédiés par con (par ex. $D \leftarrow F+G$ si $d_D + d_F + d_G \ge \varepsilon$), ou en autorisant des neutralités ($=0$) avec au moins une marge stricte.
  - La combinaison requiert des variables $(x_{p,c})$ pour $(m\text{-}1)$ et $(y_p, a_{p,c})$ pour $(1\text{-}m)$, avec des règles de non-chevauchement appropriées par con.

Conclusion: séparément, ni $(1\text{-}m)$ ni $(m\text{-}1)$ ne justifient $z \succ t$; qualitativement, une explication existe si l’on permet simultanément $B$ en $(1\text{-}m)$ (couvrant $C$) et des bundles $(m\text{-}1)$ pour $D$, $E$, sous contraintes relaxées (neutralité autorisée et au moins une compensation stricte).

## 4.2 Formulation unifiée — Explication $(1\text{-}m)$ ET $(m\text{-}1)$ combinées pour le cas général $x \succ y$

### Principe Fondamental

On cherche une explication **qui combine les deux modes** : certains cons sont couverts en $(1\text{-}m)$ (un pro couvre plusieurs cons), d'autres en $(m\text{-}1)$ (plusieurs pros couvrent un con), sans chevauchement.

**Exemple attendu pour $z \succ t$:**
- Mode $(1\text{-}m)$: Pro $B$ couvre cons $\{C, E\}$ → $d_B + d_C + d_E = 126 - 70 - 54 = +2$ ✓
- Mode $(m\text{-}1)$: Pros $\{F, G\}$ couvrent con $D$ → $d_D + d_F + d_G = -60 + 40 + 36 = +16$ ✓

### Variables (binaires)

**Mode (1-m):**
- $y_p \in \{0,1\}$: activation du pro $p$
- $a_{p,c} \in \{0,1\}$: con $c$ affecté au pro $p$ en mode $(1\text{-}m)$

**Mode (m-1):**
- $x_{p,c} \in \{0,1\}$: pro $p$ affecte son aide au con $c$ en mode $(m\text{-}1)$

**Indicateurs de partition:**
- $\text{mode}_{1\text{-}m}(c) \in \{0,1\}$: = 1 ssi le con $c$ est couvert en mode $(1\text{-}m)$
- $\text{mode}_{m\text{-}1}(c) \in \{0,1\}$: = 1 ssi le con $c$ est couvert en mode $(m\text{-}1)$

### Contraintes

#### 1. Partition stricte par con (chaque con choisit **exactement** un mode)
$$\text{mode}_{1\text{-}m}(c) + \text{mode}_{m\text{-}1}(c) = 1 \quad \forall c \in \text{Cons}$$

**Justification:** Chaque con doit être couvert (il y a une raison à $x \succ y$), mais par un seul mode pour éviter double-comptage.

#### 2. Mode $(1\text{-}m)$ : couverture et activation

Si $\text{mode}_{1\text{-}m}(c) = 1$, alors exactement un pro $p$ affecte $c$ (bijection locale):
$$\sum_{p} a_{p,c} = \text{mode}_{1\text{-}m}(c) \quad \forall c$$

Activation cohérente (on ne peut affecter que si le pro est actif):
$$a_{p,c} \le y_p \quad \forall p, c$$

**Positivité par pro en $(1\text{-}m)$** : si $y_p = 1$, alors le pro doit compenser ses cons:
$$d_p \cdot y_p + \sum_{c: a_{p,c}=1} d_c \cdot a_{p,c} \ge \varepsilon \cdot y_p \quad \forall p$$

#### 3. Mode $(m\text{-}1)$ : couverture et unicité

Si $\text{mode}_{m\text{-}1}(c) = 1$, alors au moins un pro contribue à $c$:
$$\sum_{p} x_{p,c} \ge \text{mode}_{m\text{-}1}(c) \quad \forall c$$

Unicité côté pros (chaque pro aide **au plus un** con en mode $(m\text{-}1)$):
$$\sum_{c} x_{p,c} \le 1 \quad \forall p$$

**Non-négativité par con en $(m\text{-}1)$** : chaque con doit être rendu positif par ses contributeurs:
$$d_c + \sum_{p: x_{p,c}=1} d_p \cdot x_{p,c} \ge \varepsilon \quad \forall c \text{ avec } \text{mode}_{m\text{-}1}(c) = 1$$

### Objectif

Minimiser le nombre total de pros utilisés:
$$\min \sum_{p} y_p$$

Ou maximiser la marge totale:
$$\max \left( \sum_{p} d_p y_p + \sum_{p,c} d_c a_{p,c} + \sum_{c} d_c + \sum_{p,c} d_p x_{p,c} \right)$$

### Certificat en cas d'infaisabilité

Si le modèle est infaisable, l'IIS révèle le conflit : quelles contraintes sont mutuellement incompatibles.

In [101]:
def exist_explication_mixte_1m_m1_avec_solveur(
    differences: Dict[str, float], epsilon: float = 1e-3, objective: str = "min_pros"
) -> Tuple[bool, Union[Dict, str]]:
    """
    MILP mixte (1-m) + (m-1) avec PARTITION STRICTE PAR CON + EXCLUSION PAR PRO.
    
    - Chaque cons choisit exactement un mode
    - Chaque pro ne peut être actif que dans un seul mode
    """
    pros, cons, _ = def_pros_cons_neutrals(differences)
    if len(cons) == 0:
        return True, {"mode_1m": [], "mode_m1": []}
    if len(pros) == 0:
        return False, "Aucun pro disponible."

    try:
        model = gp.Model("Explication_mixte_1m_m1")
        model.setParam('OutputFlag', 0)

        # Variables
        y = {p: model.addVar(vtype=GRB.BINARY, name=f"y_{p}") for p in pros}
        a = {(p, c): model.addVar(vtype=GRB.BINARY, name=f"a_{p}_{c}") for p in pros for c in cons}
        x = {(p, c): model.addVar(vtype=GRB.BINARY, name=f"x_{p}_{c}") for p in pros for c in cons}
        mode_1m = {c: model.addVar(vtype=GRB.BINARY, name=f"mode_1m_{c}") for c in cons}
        mode_m1 = {c: model.addVar(vtype=GRB.BINARY, name=f"mode_m1_{c}") for c in cons}

        # C1: Partition exacte par con
        for c in cons:
            model.addConstr(mode_1m[c] + mode_m1[c] == 1, name=f"partition_{c}")

        # C2: Mode (1-m) - couverture
        for c in cons:
            model.addConstr(gp.quicksum(a[(p, c)] for p in pros) == mode_1m[c], name=f"1m_cover_{c}")

        # C3: Mode (1-m) - activation
        for p in pros:
            for c in cons:
                model.addConstr(a[(p, c)] <= y[p], name=f"1m_act_{p}_{c}")

        # C4: Mode (1-m) - positivité pro
        for p in pros:
            model.addConstr(
                differences[p] * y[p] + gp.quicksum(differences[c] * a[(p, c)] for c in cons) >= epsilon * y[p],
                name=f"1m_pos_{p}"
            )

        # C5: Mode (m-1) - couverture (au moins 1 si mode_m1=1, 0 si mode_1m=1)
        for c in cons:
            model.addConstr(gp.quicksum(x[(p, c)] for p in pros) >= mode_m1[c], name=f"m1_lower_{c}")
            model.addConstr(gp.quicksum(x[(p, c)] for p in pros) <= len(pros) * mode_m1[c], name=f"m1_upper_{c}")

        # C6: Mode (m-1) - unicité pro
        for p in pros:
            model.addConstr(gp.quicksum(x[(p, c)] for c in cons) <= 1, name=f"m1_unique_{p}")

        # C7: EXCLUSION PRO - si actif en 1-m, non actif en m-1
        for p in pros:
            model.addConstr(gp.quicksum(x[(p, c)] for c in cons) <= len(cons) * (1 - y[p]), name=f"excl_{p}")

        # C8: Mode (m-1) - non-négativité con (SEULEMENT si mode_m1=1)
        M_relax = sum(abs(differences[k]) for k in differences) + 100
        for c in cons:
            model.addConstr(
                differences[c] + gp.quicksum(differences[p] * x[(p, c)] for p in pros) >= epsilon - M_relax * (1 - mode_m1[c]),
                name=f"m1_nonneg_{c}"
            )

        # Objectif
        if objective == "min_pros":
            model.setObjective(gp.quicksum(y[p] for p in pros), GRB.MINIMIZE)
        else:
            margin = (
                gp.quicksum(differences[p] * y[p] + gp.quicksum(differences[c] * a[(p, c)] for c in cons) for p in pros)
                + gp.quicksum(differences[c] + gp.quicksum(differences[p] * x[(p, c)] for p in pros) for c in cons)
            )
            model.setObjective(margin, GRB.MAXIMIZE)

        model.optimize()

        if model.status == GRB.OPTIMAL:
            mode_1m_res = []
            for p in pros:
                if y[p].X > 0.5:
                    ca = [c for c in cons if a[(p, c)].X > 0.5]
                    if ca:
                        mode_1m_res.append({"pro": p, "cons": ca})

            mode_m1_res = []
            for c in cons:
                pa = [p for p in pros if x[(p, c)].X > 0.5]
                if pa:
                    mg = differences[c] + sum(differences[p] for p in pa)
                    mode_m1_res.append({"con": c, "pros": pa, "margin": round(mg, 3)})

            return True, {"mode_1m": mode_1m_res, "mode_m1": mode_m1_res}

        elif model.status == GRB.INFEASIBLE:
            cert = "\n🔴 CERTIFICAT DE NON-EXISTENCE\n" + "=" * 70 + "\n"
            cert += "Pas d'explication mixte (1-m)+(m-1) pour cette comparaison.\n\n"
            cert += "Différences:\n"
            for cr, df in sorted(differences.items()):
                cert += f"  {cr}: {df:+.1f}\n"

            try:
                model.computeIIS()
                iis = [c.ConstrName for c in model.getConstrs() if c.IISConstr]
                if iis:
                    cert += "\nContraintes conflictuelles (IIS):\n"
                    for cn in iis:
                        cert += f"  - {cn}\n"
            except:
                pass

            cert += "\n" + "=" * 70
            return False, cert

        else:
            return False, f"Statut: {model.status}"

    except gp.GurobiError as e:
        return False, f"Erreur: {str(e)}"

In [94]:
# Exécution — Formulation unifiée pour z ≻ t
diff_zt_unif = {k: weights[k] * (candidats_q2['z'][k] - candidats_q2['t'][k]) for k in weights}
df_unif = pd.DataFrame([(k, diff_zt_unif[k]) for k in diff_zt_unif], columns=["Critère", "Différence pondérée (z - t)"])
print("\n z ≻ t (formulation unifiée)")
print(df_unif.to_string(index=False))
pros_unif, cons_unif, _ = def_pros_cons_neutrals(diff_zt_unif)
print(f"Pros: {pros_unif} | Cons: {cons_unif}")
existe_mixte, res_mixte = exist_explication_mixte_1m_m1_avec_solveur(diff_zt_unif, epsilon=1e-3, objective="min_pros")
if existe_mixte:
    print("\n✅ Explication mixte trouvée")
    if res_mixte["mode_1m"]:
        print("Mode (1-m):")
        for bloc in res_mixte["mode_1m"]:
            print(f"  - Pro {bloc['pro']} couvre {bloc['cons']}")
    if res_mixte["mode_m1"]:
        print("Mode (m-1):")
        for bloc in res_mixte["mode_m1"]:
            print(f"  - Con {bloc['con']} ← pros {bloc['pros']} (marge {bloc['margin']})")
else:
    print("\n❌ Aucune explication mixte (1-m / m-1) pour z ≻ t")
    print(res_mixte)


 z ≻ t (formulation unifiée)
Critère  Différence pondérée (z - t)
      A                            0
      B                          126
      C                          -70
      D                          -60
      E                          -54
      F                           40
      G                           36
Pros: ['B', 'F', 'G'] | Cons: ['C', 'D', 'E']

✅ Explication mixte trouvée
Mode (1-m):
  - Pro B couvre ['C', 'E']
Mode (m-1):
  - Con D ← pros ['F', 'G'] (marge 16)


---
# Validation finale
Cette section applique les différentes explications précédemment implémentées à 2 nouveaux candidats

In [113]:
"""
=== TEST DE VALIDATION FINALE ===
Comparaison de deux nouveaux candidats a1 et a2
Tous les trade-offs (1-m), (m-1), et mixte (1-m)+(m-1)
Avec APPLICATION DES COEFFICIENTS (poids) pour chaque critère
"""

# Poids des critères
weights = {'A': 8, 'B': 7, 'C': 7, 'D': 6, 'E': 6, 'F': 5, 'G': 6}

# Données des deux nouveaux candidats
scores_a1 = {'A': 89, 'B': 74, 'C': 81, 'D': 68, 'E': 84, 'F': 79, 'G': 77}
scores_a2 = {'A': 71, 'B': 84, 'C': 91, 'D': 79, 'E': 78, 'F': 73.5, 'G': 77}


# Calcul des différences pondérées
differences_a1_a2 = {crit: weights[crit] * (scores_a1[crit] - scores_a2[crit]) for crit in scores_a1.keys()}

df_contrib = pd.DataFrame(
    [
        {
            "Critère": crit,
            "Poids": weights[crit],
            "Différence pondérée": differences_a1_a2[crit],
        }
        for crit in sorted(scores_a1.keys())
    ]
)

print("=" * 80)
print("TEST DE VALIDATION: a1 ≻ a2 (TOUS LES TRADE-OFFS)")
print("=" * 80)
print(f"\nScores a1: {scores_a1}")
print(f"Scores a2: {scores_a2}")
print("\nContributions pondérées par critère:")
print(df_contrib.to_string(index=False, float_format=lambda x: f"{x:.1f}"))

# Classifier les critères
pros_a1a2, cons_a1a2, neutrals_a1a2 = def_pros_cons_neutrals(differences_a1_a2)
print(f"\nPros (a1 > a2):     {pros_a1a2}")
print(f"Cons (a1 < a2):     {cons_a1a2}")
print(f"Neutrals (a1 = a2): {neutrals_a1a2}")

# Test 1: Trade-off (1-m)
print("\n" + "=" * 80)
print("TEST 1: TRADE-OFF (1-m)")
print("=" * 80)
existe_1m_a1a2, exp_1m_a1a2 = exist_explication_1_m_avec_solveur(
    differences_a1_a2, epsilon=1e-3
)
if existe_1m_a1a2:
    print(f"✅ Explication (1-m) trouvée:")
    for matching in exp_1m_a1a2:
        print(f"  Pro {matching[0]} → Con {matching[1]} (marge: {matching[2]:.2f})")
else:
    print("❌ Pas d'explication (1-m) trouvée")

# Test 2: Trade-off (m-1) relaxé
print("\n" + "=" * 80)
print("TEST 2: TRADE-OFF (m-1) RELÂCHÉ")
print("=" * 80)
existe_m1_rel_a1a2, exp_m1_rel_a1a2 = exist_explication_m_1_relaxed_avec_solveur(
    differences_a1_a2, epsilon=1e-3
)
if existe_m1_rel_a1a2:
    print(f"✅ Explication (m-1) relâchée trouvée:")
    for con_explanation in exp_m1_rel_a1a2:
        print(f"  Con {con_explanation['con']} ← Pros {con_explanation['pros']} (marge: {con_explanation['margin']:.2f})")
else:
    print("❌ Pas d'explication (m-1) relâchée trouvée")

# Test 3: Trade-off mixte (1-m) + (m-1)
print("\n" + "=" * 80)
print("TEST 3: TRADE-OFF MIXTE (1-m) + (m-1)")
print("=" * 80)
existe_mixte_a1a2, exp_mixte_a1a2 = exist_explication_mixte_1m_m1_avec_solveur(
    differences_a1_a2, epsilon=1e-3
)
if existe_mixte_a1a2:
    print(f"✅ Explication mixte trouvée:")
    if exp_mixte_a1a2.get('mode_1m'):
        print(f"  Mode (1-m):")
        for mode_1m_item in exp_mixte_a1a2['mode_1m']:
            print(f"    Pro {mode_1m_item['pro']} couvre {mode_1m_item['cons']}")
    if exp_mixte_a1a2.get('mode_m1'):
        print(f"  Mode (m-1):")
        for mode_m1_item in exp_mixte_a1a2['mode_m1']:
            print(f"    Con {mode_m1_item['con']} ← Pros {mode_m1_item['pros']} (marge: {mode_m1_item['margin']:.2f})")
else:
    print("❌ Pas d'explication mixte trouvée")

print("\n" + "=" * 80)


TEST DE VALIDATION: a1 ≻ a2 (TOUS LES TRADE-OFFS)

Scores a1: {'A': 89, 'B': 74, 'C': 81, 'D': 68, 'E': 84, 'F': 79, 'G': 77}
Scores a2: {'A': 71, 'B': 84, 'C': 91, 'D': 79, 'E': 78, 'F': 73.5, 'G': 77}

Contributions pondérées par critère:
Critère  Poids  Différence pondérée
      A      8                144.0
      B      7                -70.0
      C      7                -70.0
      D      6                -66.0
      E      6                 36.0
      F      5                 27.5
      G      6                  0.0

Pros (a1 > a2):     ['A', 'E', 'F']
Cons (a1 < a2):     ['B', 'C', 'D']
Neutrals (a1 = a2): ['G']

TEST 1: TRADE-OFF (1-m)
❌ Pas d'explication (1-m) trouvée

TEST 2: TRADE-OFF (m-1) RELÂCHÉ
❌ Pas d'explication (m-1) relâchée trouvée

TEST 3: TRADE-OFF MIXTE (1-m) + (m-1)
❌ Pas d'explication mixte trouvée

